# M02. 결측치 처리

> 📌 **언제 필요한가**  
> 받은 데이터에 빈 칸(`NaN`)이 있을 때. 거의 모든 진짜 데이터에 결측치가 있어요.

## 이 모듈에서 배울 것

- 결측치 확인 방법 (`isnull`, `isna`)
- 결측치 제거 (`dropna`, axis=0 vs axis=1)
- 결측치 채우기 (`fillna`)
- 어떤 방법을 언제 쓸지 판단

---


## 📥 데이터 준비

> 이 모듈은 아래 파일이 필요해요. **저장소에는 동봉돼 있지 않으니** 먼저 받아서 두세요.

- `OBS_ASOS_TIM.csv` — 기상청 종관기상관측(ASOS) **인천** 시간자료 · 인코딩 `cp949`

### 받는 곳

[기상자료개방포털 — 종관기상관측(ASOS) 자료](https://data.kma.go.kr/data/grnd/selectAsosRltmList.do?pgmNo=36)

### 받는 방법

1. 위 링크 접속 → **자료형태**를 `시간 자료`로 선택
2. **기간** 지정 (예: `2026-06-10 01시` ~ `2026-06-11 00시`)
3. **지점**: 지도/목록에서 **인천광역시 → 인천** 체크
4. **요소**: `기온`, `강수량`, `풍속`, `습도` 체크
5. **조회** → **CSV 다운로드**

![검색 조건 설정 화면](images/m02_asos_search.png)

> ⚠️ **비회원은 일부만 받아져요** — 일자료 31건 / 시간자료 24건까지. 그래서 이 파일은 **24행**이에요. (전체가 필요하면 로그인)

![비회원 다운로드 안내 팝업](images/m02_asos_login.png)

### 둘 때 주의

- 받은 파일명이 `OBS_ASOS_TIM_20260612010409.csv`처럼 **타임스탬프**가 붙어 있으면 → **`OBS_ASOS_TIM.csv`** 로 바꿔 저장하세요. (코드가 이 이름을 기대해요)
- 두는 곳 — **로컬 Jupyter**: `data/` 폴더 / **Colab**: `/content/`에 업로드.

> 💡 이 데이터의 **`강수량(mm)`** 컬럼은 그 기간에 비가 안 와서 **24시간 모두 비어 있어요(`NaN`)**. 결측치 실습에 딱 맞는 자연 결측이에요.

컬럼 설명·함정 등 자세한 내용은 [`data/README.md`](data/README.md) 참고.

---

## 1. 결측치 있는지 확인하기

먼저 데이터에 결측치가 있는지부터 봐야 해요.


In [ ]:
import pandas as pd

weather = pd.read_csv('data/OBS_ASOS_TIM.csv', encoding='cp949')
weather.shape

In [ ]:
weather.head()

### 결측치 개수 확인 - 두가지 방법

In [ ]:
weather.isnull().sum()

In [ ]:
weather.isna().sum()


> 💡 `isnull()`과 `isna()`는 완전히 같은 메서드예요. 어떤 걸 써도 OK.


## 2. 결측치 제거 — `dropna`

### 2.1 결측치 있는 **행** 제거 (axis=0, 기본)


In [ ]:
# axis=0이 기본 (생략 가능)
weather_no_na_rows = weather.dropna()
print(f"원래: {weather.shape}")
print(f"행 제거 후: {weather_no_na_rows.shape}")
weather_no_na_rows


### 2.2 결측치 있는 **컬럼** 제거 (axis=1)


In [ ]:
# axis=1: 결측치가 있는 컬럼 통째로 제거
weather_no_na_cols = weather.dropna(axis=1)
print(f"원래: {weather.shape}")
print(f"컬럼 제거 후: {weather_no_na_cols.shape}")
weather_no_na_cols.head()


> 💡 **언제 axis=0, 언제 axis=1?**
> - **컬럼 전체가 NaN**이면 axis=1로 컬럼 삭제 (이번 데이터의 `강수량(mm)` 컬럼 — 그 기간 비가 안 와서 24시간 모두 NaN)
> - **일부 행만 NaN**이면 axis=0으로 그 행만 삭제
>
> ⚠️ 이번 데이터처럼 **한 컬럼이 통째로 NaN**이면 `dropna()`(axis=0)는 **모든 행을 지워버려요** (24행 → 0행). 그래서 여기선 `axis=1`이 맞아요.


## 3. 결측치 채우기 — `fillna`

제거 대신 값으로 채우는 방법.


In [ ]:
# 예시: 일부러 결측치 만들어서 채워보기
import numpy as np

example = pd.DataFrame({
    '이름': ['철수', '영희', '민수', '지영'],
    '점수': [85, np.nan, 70, np.nan],
    '학년': [1, 2, np.nan, 3]
})
print("원본:")
print(example)


In [ ]:
# 방법 1: 특정 값으로 채우기 (예: 0)
example.fillna(0)


In [ ]:
# 방법 2: 평균으로 채우기 (수치 컬럼)
example_filled = example.copy()
example_filled['점수'] = example['점수'].fillna(example['점수'].mean())
example_filled


In [ ]:
# 방법 3: 컬럼마다 다른 값으로
example.fillna({'점수': example['점수'].mean(), '학년': 1})


## 4. 결측치 처리 결정 가이드

| 상황 | 추천 방법 |
|---|---|
| 컬럼 전체가 NaN | `dropna(axis=1)` |
| 결측 행이 전체의 5% 미만 | `dropna()` |
| 결측 행이 많고 분포가 의미 있음 | `fillna(평균/중앙값)` |
| 카테고리 변수 결측 | `fillna('unknown')` 또는 `dropna` |
| 시계열 데이터 | `ffill()` (앞 값으로) |

> 💡 **결측치 처리는 분석에 영향을 줘요.** 어떤 방법을 썼는지 항상 기록하세요.


## 5. 본인 데이터에 적용해보기 ✏️


In [ ]:
# 본인 데이터로 시도
# my_df = pd.read_csv('본인_파일.csv', encoding='cp949')
# 
# # Step 1: 결측치 개수 확인
# print(my_df.isnull().sum())
# 
# # Step 2: 시각화로 결측 패턴 보기
# import seaborn as sns
# sns.heatmap(my_df.isnull(), cbar=False, yticklabels=False)
# 
# # Step 3: 적절한 방법으로 처리
# my_df_clean = my_df.dropna()  # 또는 fillna


## 6. ⚠️ 주의사항

### 6.1 `dropna()`는 원본을 바꾸지 않음
```python
df.dropna()       # 결과 반환 (원본 그대로)
df = df.dropna()  # 원본 갱신
df.dropna(inplace=True)  # 원본 직접 수정 (권장 X)
```

### 6.2 평균 대체는 분산을 줄임
모든 결측을 평균으로 채우면 그 변수의 분산이 인위적으로 줄어들어요. 통계 분석 시 조심.

### 6.3 결측에도 의미가 있을 수 있음
"응답 안 함"도 정보일 수 있어요. 무작정 제거하지 말고 **왜 빠졌는지** 먼저 생각하세요.

### 6.4 결측치가 NaN이 아닐 수도
일부 데이터는 `-`, `999`, `9999`, 빈 문자열로 결측을 표시. `isnull()`로 안 잡혀요.  
**해결**: `df.replace([-1, 999], np.nan)` 식으로 변환 후 처리.


## 7. 📚 더 알아보기

- `df.isnull().sum().sum()` — 전체 결측 개수
- `df.dropna(thresh=5)` — 비결측이 5개 이상인 행/열만 유지
- `df.interpolate()` — 보간으로 채우기 (수치 시계열)
- 시각화: `seaborn.heatmap(df.isnull())` 또는 `missingno` 라이브러리
